In [ ]:
img_path= 'DukeData/train/images/Subject_02_01.npy'

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
img = np.load(img_path)

In [ ]:
plt.imshow(img)
plt.title("OCT Image")
plt.colorbar()
plt.show()

In [ ]:
"""pip install opencv-python"""

In [ ]:
from sklearn.cluster import KMeans


pixels = img.flatten().reshape(-1, 1) # Flatten image for clustering
kmeans = KMeans(n_clusters=2).fit(pixels)
segmented_labels = kmeans.labels_.reshape(img.shape)

# Extract ROI and background based on cluster labels
roi = img[segmented_labels == 1]
background = img[segmented_labels == 0]


In [ ]:
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.imshow(img, cmap='gray')
plt.title('Original Image')
plt.colorbar()

#
plt.subplot(1, 2, 2)
plt.imshow(segmented_labels, cmap='jet')
plt.title('Segmented Image')
plt.colorbar()

plt.tight_layout()
plt.show()

In [ ]:
import os
import numpy as np
from sklearn.cluster import KMeans


folder_path = 'DukeData/train/images'


output_file = "cnr_results.txt"


cnr_results = []

# Iterate through all files in the folder
for file_name in os.listdir(folder_path):
    if file_name.endswith(".npy"):
        file_path = os.path.join(folder_path, file_name)

        img = np.load(file_path)
        print(f"Processing {file_name}...")
        # Flatten image for clustering
        pixels = img.flatten().reshape(-1, 1)

        kmeans = KMeans(n_clusters=2, n_init=10, random_state=42).fit(pixels)
        segmented_labels = kmeans.labels_.reshape(img.shape)

        roi = img[segmented_labels == 1]
        background = img[segmented_labels == 0]

        S1 = np.mean(roi)
        S2 = np.mean(background)

        # Calculate noise (standard deviation of background)
        back_sigma = np.std(background)
        roi_sigma = np.std(roi)


        if back_sigma != 0 or roi_sigma!=0:  # Avoid division by zero
            CNR = abs(S1 - S2) / np.sqrt(roi_sigma**2 + back_sigma**2)
        else:
            CNR = float('inf')  # Assign infinity if sigma is 0

        cnr_results.append((file_name, CNR))

        print(f"File: {file_name}, CNR: {CNR}")


with open(output_file, "w") as f:
    f.write("File Name\tCNR\n")
    for file_name, CNR in cnr_results:
        f.write(f"{file_name}\t{CNR:.4f}\n")

print(f"CNR calculations completed. Results saved to {output_file}.")


In [ ]:
import csv
with open("cnr_results.csv", "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["File Name", "CNR"])
    writer.writerows(cnr_results)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("white")
cnr_values = [cnr for _, cnr in cnr_results]
plt.figure(figsize=(6, 3))
plt.hist(cnr_values, bins=10, alpha=0.7)
#plt.title("CNR Distribution for Duke dataset")
plt.xlim(0, 15)
plt.ylim(0, 100)
plt.xlabel("CNR", fontsize=20)
plt.ylabel("Frequency", fontsize=20)
plt.xticks(fontsize=20)
plt.yticks(fontsize=20)
sns.despine()
output_path = 'CNR_Kmeans_Duke.pdf'
#plt.axis('off')
plt.savefig(output_path, format='pdf',bbox_inches='tight', dpi=300)
plt.show()






In [ ]:
import os
import numpy as np
from sklearn.cluster import KMeans


folder_path = 'UMNData/train/images'


#output_file = "cnr_results.txt"


cnr_results = []

# Iterate through all files in the folder
for file_name in os.listdir(folder_path):
    if file_name.endswith(".npy"):
        file_path = os.path.join(folder_path, file_name)

        img = np.load(file_path)
        print(f"Processing {file_name}...")
        # Flatten image for clustering
        pixels = img.flatten().reshape(-1, 1)

        kmeans = KMeans(n_clusters=2, n_init=10, random_state=42).fit(pixels)
        segmented_labels = kmeans.labels_.reshape(img.shape)

        roi = img[segmented_labels == 1]
        background = img[segmented_labels == 0]

        S1 = np.mean(roi)
        S2 = np.mean(background)

        # Calculate noise (standard deviation of background)
        back_sigma = np.std(background)
        roi_sigma = np.std(roi)


        if back_sigma != 0 or roi_sigma!=0:  # Avoid division by zero
            CNR = abs(S1 - S2) / np.sqrt(roi_sigma**2 + back_sigma**2)
        else:
            CNR = float('inf')  # Assign infinity if sigma is 0

        cnr_results.append((file_name, CNR))

"""        print(f"File: {file_name}, CNR: {CNR}")


with open(output_file, "w") as f:
    f.write("File Name\tCNR\n")
    for file_name, CNR in cnr_results:
        f.write(f"{file_name}\t{CNR:.4f}\n")

print(f"CNR calculations completed. Results saved to {output_file}.")"""


In [ ]:
import matplotlib.pyplot as plt


sns.set_style("white")
cnr_values = [cnr for _, cnr in cnr_results]
plt.figure(figsize=(6, 3))
plt.hist(cnr_values, bins=10, alpha=0.7)
#plt.title("CNR Distribution for UMN dataset")
plt.xlim(0, 15)
plt.xticks(fontsize=20)
plt.yticks(fontsize=20)
plt.xlabel("CNR", fontsize=20)
plt.ylabel("Frequency", fontsize=20)
plt.xlabel("CNR")
sns.despine()
output_path = 'CNR_Kmeans_UMN.pdf'
#plt.axis('off')
plt.savefig(output_path, format='pdf',bbox_inches='tight', dpi=300)
plt.show()




In [ ]:
# theresholding


import numpy as np
import matplotlib.pyplot as plt

import cv2


def apply_otsu_thresholding_to_npy(npy_file_path):
  """
  Applies Otsu's thresholding to an image loaded from an .npy file.

  Args:
    npy_file_path: Path to the .npy file containing the image data.

  Returns:
    A tuple containing:
      - The calculated threshold value.
      - The thresholded image as a NumPy array.
  """


  img_array = np.load(npy_file_path)


  img = img_array


  img_uint8 = (img * 255).astype(np.uint8)

  # Apply Otsu's thresholding
  ret, thresholded_img = cv2.threshold(img_uint8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

  return ret, thresholded_img,img_uint8




folder_path = 'UMNData/train/images'




cnr_results = []

# Iterate through all files in the folder
for file_name in os.listdir(folder_path):
    if file_name.endswith(".npy"):
        file_path = os.path.join(folder_path, file_name)
        threshold_value, thresholded_image ,image= apply_otsu_thresholding_to_npy(file_path)
        print(f"Otsu's Threshold Value for {file_name}: {threshold_value}")
        """plt.figure(figsize=(10, 5))

        # Original Image
        plt.subplot(1, 2, 1)
        plt.title("Original Image")
        plt.imshow(image, cmap='gray')
        plt.axis('off')

        # Thresholded Image
        plt.subplot(1, 2, 2)
        plt.title("Thresholded Image (Otsu's)")
        plt.imshow(thresholded_image, cmap='gray')
        plt.axis('off')

        plt.show()"""
        signal_region = image[image >= threshold_value]  # Pixels >= threshold
        noise_region = image[image < threshold_value]   # Pixels < threshold


        if len(signal_region) > 0 and len(noise_region) > 0:  # Avoid division by zero
            mu_s = np.mean(signal_region)
            sigma_s = np.std(signal_region)
            mu_n = np.mean(noise_region)
            sigma_n = np.std(noise_region)

            cnr = abs(mu_s - mu_n) / np.sqrt(sigma_s**2 + sigma_n**2)
        else:
            cnr = 0  # Default to 0 if one of the regions is empty
        cnr_results.append(cnr)











In [ ]:
 # plot one sample from UMN dataset

 plt.figure(figsize=(10, 5))


plt.subplot(1, 2, 1)
plt.title("Original Image")
plt.imshow(image, cmap='gray')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.title("Thresholded Image (Otsu's)")
plt.imshow(thresholded_image, cmap='gray')
plt.axis('off')

plt.show()

In [ ]:
import matplotlib.pyplot as plt

cnr_values = [cnr for cnr in cnr_results]
plt.figure(figsize=(6, 3))
plt.hist(cnr_values, bins=10, alpha=0.7)
#plt.title("CNR Distribution for UMN dataset")
plt.xlabel("CNR")
plt.xlim(0, 15)
plt.ylim(0,160)
plt.xticks(fontsize=20)
plt.yticks(fontsize=20)
plt.xlabel("CNR", fontsize=20)
plt.ylabel("Frequency", fontsize=20)
plt.xlabel("CNR")
sns.despine()
output_path = 'CNR_Thr_UMN.pdf'
#plt.axis('off')
plt.savefig(output_path, format='pdf',bbox_inches='tight', dpi=300)
plt.show()

plt.show()



In [ ]:
# theresholding


import numpy as np
import matplotlib.pyplot as plt

import cv2


def apply_otsu_thresholding_to_npy(npy_file_path):
  """
  Applies Otsu's thresholding to an image loaded from an .npy file.

  Args:
    npy_file_path: Path to the .npy file containing the image data.

  Returns:
    A tuple containing:
      - The calculated threshold value.
      - The thresholded image as a NumPy array.
  """


  img_array = np.load(npy_file_path)


  img = img_array


  img_uint8 = (img * 255).astype(np.uint8)

  # Apply Otsu's thresholding
  ret, thresholded_img = cv2.threshold(img_uint8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

  return ret, thresholded_img,img_uint8




folder_path = 'DukeData/train/images'




cnr_results_duke = []

# Iterate through all files in the folder
for file_name in os.listdir(folder_path):
    if file_name.endswith(".npy"):
        file_path = os.path.join(folder_path, file_name)
        threshold_value, thresholded_image ,image= apply_otsu_thresholding_to_npy(file_path)
        print(f"Otsu's Threshold Value for {file_name}: {threshold_value}")
        """plt.figure(figsize=(10, 5))

        # Original Image
        plt.subplot(1, 2, 1)
        plt.title("Original Image")
        plt.imshow(image, cmap='gray')
        plt.axis('off')

        # Thresholded Image
        plt.subplot(1, 2, 2)
        plt.title("Thresholded Image (Otsu's)")
        plt.imshow(thresholded_image, cmap='gray')
        plt.axis('off')

        plt.show()"""
        signal_region = image[image >= threshold_value]  # Pixels >= threshold
        noise_region = image[image < threshold_value]   # Pixels < threshold


        if len(signal_region) > 0 and len(noise_region) > 0:  # Avoid division by zero
            mu_s = np.mean(signal_region)
            sigma_s = np.std(signal_region)
            mu_n = np.mean(noise_region)
            sigma_n = np.std(noise_region)

            cnr = abs(mu_s - mu_n) / np.sqrt(sigma_s**2 + sigma_n**2)
        else:
            cnr = 0  # Default to 0 if one of the regions is empty
        cnr_results_duke.append(cnr)











In [ ]:
 # plot one sample from UMN dataset

 plt.figure(figsize=(10, 5))


plt.subplot(1, 2, 1)
plt.title("Original Image")
plt.imshow(image, cmap='gray')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.title("Thresholded Image (Otsu's)")
plt.imshow(thresholded_image, cmap='gray')
plt.axis('off')

plt.show()

In [ ]:
import matplotlib.pyplot as plt

cnr_values = [cnr for cnr in cnr_results_duke]
plt.figure(figsize=(6, 3))
plt.hist(cnr_values, bins=10, alpha=0.7)
plt.title("CNR Distribution for Duke dataset")
plt.xlabel("CNR")
plt.xlim(0, 15)
plt.ylim(0, 160)
plt.xticks(fontsize=20)
plt.yticks(fontsize=20)
plt.xlabel("CNR", fontsize=20)
plt.ylabel("Frequency", fontsize=20)
plt.xlabel("CNR")
sns.despine()
output_path = 'CNR_thr_duke.pdf'
#plt.axis('off')
plt.savefig(output_path, format='pdf',bbox_inches='tight', dpi=300)
plt.show()

plt.show()






In [ ]:
# during train
import pandas as pd
df=pd.read_csv('model_results_fixed_epsilon_mae.csv')

In [ ]:
df= df.iloc[-1,:]

In [ ]:
df.columns

In [ ]:
type(df['Validation_Dice'])

In [ ]:
s= df['per_layer_all_list']

In [ ]:
import ast

my_list = ast.literal_eval(s)
print(my_list)
print(type(my_list))
